In [1]:
# pip install langchain langchain_openai langchain_community chromadb

In [2]:
# from google.colab import drive
# import os

# # 먼저 구글 드라이브 마운트
# drive.mount('/content/drive')

In [3]:
import os
from dotenv import load_dotenv

# .env 파일에서 환경 변수 로드
load_dotenv("/content/.env")

# 환경 변수에서 API 키 가져오기
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 로더 설정
loaders = [TextLoader("D:\WorkSpace\Python\langchain-tutorial\Ch04. Advanced Rag\Data\How_to_invest_money.txt", encoding="utf-8")]

docs = []
for loader in loaders:
    docs.extend(loader.load())


d:\WorkSpace\Python\langchain-tutorial\Ch04. Advanced Rag\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# 문서 생성을 위한 텍스트 분할기 정의
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# 문서 분할
split_docs = recursive_splitter.split_documents(docs)

# OpenAIEmbeddings 인스턴스 생성
embeddings = OpenAIEmbeddings()

# Chroma vectorstore 생성
vectorstore = Chroma.from_documents(documents=split_docs, embedding=embeddings)

# Chroma vectorstore 기반 리트리버 생성
retriever = vectorstore.as_retriever()

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_text_splitters import CharacterTextSplitter


# 1. 가상 문서 생성 체인
def create_virtual_doc_chain():
    system = "당신은 고도로 숙련된 AI입니다."
    user = """
    주어진 질문 '{query}'에 대해 직접적으로 답변하는 가상의 문서를 생성하세요.
    문서의 크기는 {chunk_size} 글자 언저리여야 합니다.
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human", user)
    ])
    llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
    return prompt | llm | StrOutputParser()

In [7]:
# 2. 문서 검색 체인
def create_retrieval_chain():
    return RunnableLambda(lambda x: retriever.invoke(x['virtual_doc']))

# 유틸리티 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [8]:
# 3. 최종 응답 생성 체인
def create_final_response_chain():
    final_prompt = ChatPromptTemplate.from_template("""
    다음 정보와 질문을 바탕으로 답변해주세요:

    컨텍스트: {context}

    질문: {question}

    답변:
    """)
    final_llm = ChatOpenAI(model="gpt-4o", temperature=0.2)
    return final_prompt | final_llm

In [9]:
from langchain_core.runnables import RunnableLambda

def print_input_output(input_data, output_data, step_name):
    print(f"\n--- {step_name} ---")
    print(f"Input: {input_data}")
    print(f"Output: {output_data}")
    print("-" * 50)

In [10]:
def create_pipeline_with_logging():
    virtual_doc_chain = create_virtual_doc_chain()
    retrieval_chain = create_retrieval_chain()
    final_response_chain = create_final_response_chain()

    # 가상 문서 생성 단계
    def virtual_doc_step(x):
        result = {"virtual_doc": virtual_doc_chain.invoke({
            "query": x["question"],
            "chunk_size": 200
        })}
        print_input_output(x, result, "Virtual Doc Generation")
        return {**x, **result}

    # 문서 검색 단계
    def retrieval_step(x):
        result = {"retrieved_docs": retrieval_chain.invoke(x)}
        print_input_output(x, result, "Document Retrieval")
        return {**x, **result}

    # 컨텍스트 포맷팅 단계
    def context_formatting_step(x):
        result = {"context": format_docs(x["retrieved_docs"])}
        print_input_output(x, result, "Context Formatting")
        return {**x, **result}


    # 최종 응답 생성 단계
    def final_response_step(x):
        result = final_response_chain.invoke(x)
        print_input_output(x, result, "Final Response Generation")
        return result

    # 전체 파이프라인 구성
    pipeline = (
        RunnableLambda(virtual_doc_step)
        | RunnableLambda(retrieval_step)
        | RunnableLambda(context_formatting_step)
        | RunnableLambda(final_response_step)
    )

    return pipeline

# 파이프라인 객체 생성
pipeline = create_pipeline_with_logging()

In [11]:
# 예시 질문과 답변
question = "주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?"
response = pipeline.invoke({"question": question})
print(f"최종 답변: {response.content}")


--- Virtual Doc Generation ---
Input: {'question': '주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?'}
Output: {'virtual_doc': '주식 시장의 변동성이 높을 때 투자 전략은 신중한 접근이 필요합니다. 첫째, 포트폴리오를 다각화하여 리스크를 분산시키는 것이 중요합니다. 둘째, 방어적인 주식이나 배당주에 투자하여 안정성을 확보할 수 있습니다. 셋째, 현금 비중을 늘려 기회를 기다리는 것도 전략입니다. 마지막으로, 장기적인 관점에서 시장의 일시적 변동성을 견디는 인내심이 필요합니다. 이러한 전략은 변동성 속에서도 투자자의 자산을 보호하고 성장 기회를 찾는 데 도움이 됩니다.'}
--------------------------------------------------

--- Document Retrieval ---
Input: {'question': '주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?', 'virtual_doc': '주식 시장의 변동성이 높을 때 투자 전략은 신중한 접근이 필요합니다. 첫째, 포트폴리오를 다각화하여 리스크를 분산시키는 것이 중요합니다. 둘째, 방어적인 주식이나 배당주에 투자하여 안정성을 확보할 수 있습니다. 셋째, 현금 비중을 늘려 기회를 기다리는 것도 전략입니다. 마지막으로, 장기적인 관점에서 시장의 일시적 변동성을 견디는 인내심이 필요합니다. 이러한 전략은 변동성 속에서도 투자자의 자산을 보호하고 성장 기회를 찾는 데 도움이 됩니다.'}
Output: {'retrieved_docs': [Document(metadata={'source': 'D:\\WorkSpace\\Python\\langchain-tutorial\\Ch04. Advanced Rag\\Data\\How_to_invest_money.txt'}, page_content='For the successful investment of money, however, a good deal more is\n